In [69]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [71]:
# all libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime
import joblib
import lightgbm as lgb 
plt.style.use('seaborn-v0_8-darkgrid')

from src.metrics import wrmsse, calculate_mape, calculate_wape, bias
from src.features import GetLagRollFeatures , get_avg_sales, get_price_features, get_trend_features
from src.utils import get_items_top, get_items_with_min_history
from src.pipeline import recursive_forecast_batch
from train_and_eval import train_models
import gc 
import warnings

In [72]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR/"data"/"processed"

parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

### Model stability on the future unknown datasets

* model: lgb

* training period: 2 years
* items: full items in the last training period with atleast 100 days of history
* testing period: 137 days

In [73]:
models_dir = BASE_DIR/'models' 

model_point = joblib.load(models_dir/"lgb_ca1_2yr_point.pkl")
model_q10 = joblib.load(models_dir/'lgb_ca1_2yr_q10.pkl')
model_q90 = joblib.load(models_dir/'lgb_ca1_2yr_q90.pkl')
models  = {'point':model_point,'q10':model_q10,'q90':model_q90}

items_unique = pd.read_pickle(models_dir/"items_min_100days_ca1.pkl")['items']

future_data = pd.read_parquet(DATA_DIR/'sales_future_ca_1.parquet')
known_data = pd.read_parquet(DATA_DIR/'sales_known_ca_1.parquet')
final_train_data =  pd.read_parquet(DATA_DIR/'final_training_ca1.parquet')

unique_features = pd.read_pickle(models_dir/'feature_cols_recursive_ca1.pkl')

#future data with only selected items
# training_data = known_data[known_data['item_id'].isin(items_unique)]
future_selected_items = future_data[future_data['item_id'].isin(items_unique)]

print(f"training data: {final_train_data['item_id'].unique()}, \n future data: {future_selected_items['item_id'].unique()}")
# items_373

training data: ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', 'FOODS_1_005', ..., 'HOUSEHOLD_2_512', 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']
Length: 3043
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516'], 
 future data: ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', 'FOODS_1_005', ..., 'HOUSEHOLD_2_512', 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']
Length: 3043
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']


In [74]:
min_date = final_train_data['date'].min()
max_date = final_train_data['date'].max()
print(max_date,min_date,(max_date-min_date).days)

2016-01-05 00:00:00 2014-01-05 00:00:00 730


In [75]:
cat_categories = {}
CAT_COLS = ['item_id', 'dept_id', 'cat_id']
for col in CAT_COLS:
    cats = sorted(final_train_data[col].dropna().unique().tolist())
    final_train_data[col] = pd.Categorical(final_train_data[col], categories=cats)
    cat_categories[col] = cats
# cat_categories

### Recursive on future data: (137 days)
*  we can find stress test in 4 different windows to find at which point error become large and a recaliberation or reanchoring of training window is needed.
    1. with full data in one window (probably error will be higher, so does WRMSSE)
    2. with a 3 month window 
    3. with a 2 month window
    4. with a 1 month window 

In [76]:
dynamic_cols = [feat for feat in unique_features if 'lag' in feat or 'roll' in feat] + ['selling_trend','demand_vs_historical_mean']

In [77]:
dynamic_cols

['lag_28',
 'rolling_mean_7',
 'rolling_max_7',
 'lag_90',
 'rolling_max_60',
 'rolling_lag_28_win_7',
 'lag_60',
 'rolling_lag_28_win_28',
 'rolling_max_90',
 'rolling_mean_28',
 'rolling_mean_90',
 'rolling_max_28',
 'lag_7',
 'rolling_mean_60',
 'selling_trend',
 'demand_vs_historical_mean']

In [78]:
static_cols = list(set(unique_features) - set(dynamic_cols))
static_cols

['weekday',
 'day_of_week',
 'event_name_2',
 'is_discounted',
 'price_ratio_mean',
 'snap_CA',
 'historical_mean',
 'event_type_1',
 'historical_std',
 'week_of_year',
 'wday',
 'dept_id',
 'event_name_1',
 'year',
 'day_of_month',
 'event_type_2',
 'month',
 'wm_yr_wk',
 'cat_id',
 'sell_price',
 'item_id',
 'delta_price_rltv_dept']

In [102]:
def stress_testing_expt(trained_models,train_df,test_df,features,static_features,
                        window_length:int=28,retrain=False,cat_categories=cat_categories):
    '''stress test trained model on future data (test_df) in given windows of length(window_length)
    with possibility of retraining model every windows (retrain: True  or False) '''
    get_lag_roll_features = GetLagRollFeatures()
    fixed_training_base = train_df.copy()
    raw_history = train_df[static_features+['date','sales']].copy()
    # num_windows = int((test_df['date'].max()-test_df['date'].min()).days/window_length)
    num_windows = 0

    active_models =  trained_models  
    # keep a dict for metrics
    window_scores = {}
    bias_scores = {}
    preds = {}

    future_start = test_df['date'].min()
    for i in range(num_windows+1):
        future_end = future_start + pd.Timedelta(days=window_length)
        if future_end > test_df['date'].max():
            future_end = test_df['date'].max()

        mask_window = test_df['date'].between(future_start,future_end)
        future_window = test_df.loc[mask_window].copy() 
        
        print(f"\n === Window{i+1}: {future_start.date()} to {future_end.date()} ===")

        # generate recursive forecast for the given window length (default 28)
        pred_window, _ = recursive_forecast_batch(models=active_models,history_df=raw_history,future_static_df=
                                                  future_window,feature_cols=features,
                                                  cat_categories=cat_categories)
        preds[f'window_{i+1}'] = pred_window
       
        # WRMSSE calculated against fixed training base
        wrmsse_score = wrmsse(fixed_training_base,future_window,pred_window)
        bias_score = bias(future_window['sales'],pred_window['sales_pred'])

        window_scores[f'window_{i+1}'] = wrmsse_score
        bias_scores[f'window_{i+1}'] = bias_score
        print(f"WRMSSE: {wrmsse_score} | BIAS: {bias_score}")

        if retrain:
            #  1. select static features cols from future window
            actuals = future_window[raw_history.columns].copy()

            # make sure that actuals and raw_history has exact same columns
            assert set(actuals.columns) == set(raw_history.columns), (f"Column mismatch!\n"f"Actual columns: {list(actuals.columns)}\n\
                                                                    Raw history columns: {list(raw_history.columns)}")
            # Append predicted actuals to raw sales history for retraining
            raw_history = pd.concat([raw_history,actuals],ignore_index=True)

            # build lag, rolling and trend features for this new training data
            new_train_features = get_lag_roll_features.add_lags(raw_history,lags=[1,7,28,60,90])
            new_train_features = get_lag_roll_features.add_rolling_mean(new_train_features,windows=[7,28,60,90])
            new_train_features = get_lag_roll_features.add_rolling_max(new_train_features,windows=[7,28,60,90])
            new_train_features = get_lag_roll_features.add_rolling_on_lag(new_train_features,lags=[28],windows=[7,28])
            new_train_features = get_trend_features(new_train_features)

            # retrain and update the models
            retrained_models, _,_ = train_models(new_train_features,quantiles=True,feature_cols=features,
                                                 n_estimators=200)
            active_models = retrained_models

        else:
             # static model mode: update history with PREDICTIONS to feed lags into window i+1
            pred_to_append = future_window[static_features+['date']].copy() # to merge with raw history, need exact columns
            pred_to_append = pred_to_append.merge(pred_window[['item_id','date','sales_pred']],on=['item_id','date'],
                                                    how='inner').rename(columns={'sales_pred':'sales'})
    
            # align the columns with raw_history
            pred_to_append = pred_to_append[raw_history.columns]
    
            # append predictions to  raw_history so lag_7, rolling_mean_28 use forecasted values not real actual values
            raw_history = pd.concat([raw_history,pred_to_append],ignore_index=True)
            
        # 4. reset the start of future window
        future_start = future_end
        
    return window_scores,bias_score,preds


In [82]:
# first we have to calculate price features for the future dataset , for that first concat train data with future
# data, with only basic columns and then create price features
cols = list(future_data.columns)

train_data = final_train_data[cols].copy()
test_data = future_data[cols].copy()

cutoff  = train_data['date'].max()

# concat together
full_data = pd.concat((train_data,test_data),ignore_index=True)
# get the price features 
full_data = get_price_features(full_data)
test_data_price = full_data[full_data['date']>cutoff]
# merge with original test
# final_future_data = future_data.merge(test_data_price,on=['item_id','store_id','dept_id','wm_yr_wk','date'],how='left')
test_data_price.columns


Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week',
       'day_of_month', 'week_of_year', 'delta_price_weekn-1',
       'historical_mean', 'historical_std', 'price_ratio_mean',
       'is_discounted', 'dept_mean_price', 'delta_price_rltv_dept'],
      dtype='object')

In [89]:

CAT_COLS = ['item_id', 'dept_id', 'cat_id','event_type_1','event_type_2','event_name_1','event_name_2']
cat_categories = {}
for col in CAT_COLS:
    cats = sorted(test_data_price[col].dropna().unique().tolist())
    test_data_price[col] = pd.Categorical(test_data_price[col], categories=cats)
    cat_categories[col] = cats

In [92]:
results_win_28 = stress_testing_expt(trained_models=models,train_df=final_train_data,test_df=test_data_price,
                                     features=unique_features,static_features=static_cols,window_length=28,
                                     retrain=False,cat_categories=cat_categories)


 === Window1: 2016-01-06 to 2016-02-03 ===
WRMSSE: 0.8443739790676126 | BIAS: 0.16565732956930068

 === Window2: 2016-02-03 to 2016-03-02 ===
WRMSSE: 0.9972500098367449 | BIAS: 0.3317230905456763

 === Window3: 2016-03-02 to 2016-03-30 ===
WRMSSE: 1.1099135608878017 | BIAS: 0.4156045675778765

 === Window4: 2016-03-30 to 2016-04-27 ===
WRMSSE: 1.2027666475975582 | BIAS: 0.45909074413808393

 === Window5: 2016-04-27 to 2016-05-22 ===
WRMSSE: 1.2721468279153723 | BIAS: 0.4826512998700567


In [95]:
# NOW WITH 2 MONTHS 
results_win_60 = stress_testing_expt(trained_models=models,train_df=final_train_data,test_df=test_data_price,
                                     features=unique_features,static_features=static_cols,window_length=60,
                                     retrain=False,cat_categories=cat_categories)


 === Window1: 2016-01-06 to 2016-03-06 ===
WRMSSE: 0.9586984234155531 | BIAS: 0.23477743546545496

 === Window2: 2016-03-06 to 2016-05-05 ===
WRMSSE: 1.2131503293553354 | BIAS: 0.45449949611533363

 === Window3: 2016-05-05 to 2016-05-22 ===
WRMSSE: 1.2880092013527018 | BIAS: 0.5302581859742207


In [100]:
# WITH RETRAINING: 
results_win_28_retrain = stress_testing_expt(trained_models=models,train_df=final_train_data,test_df=test_data_price,
                                     features=unique_features,static_features=static_cols,window_length=28,
                                     retrain=True,cat_categories=cat_categories)


 === Window1: 2016-01-06 to 2016-02-03 ===
WRMSSE: 0.8443739790676126 | BIAS: 0.16565732956930068
quantile models training.

 === Window2: 2016-02-03 to 2016-03-02 ===
WRMSSE: 0.8652163327538142 | BIAS: 0.22969447434099882
quantile models training.

 === Window3: 2016-03-02 to 2016-03-30 ===
WRMSSE: 0.8418651303597213 | BIAS: 0.22683179298315348
quantile models training.

 === Window4: 2016-03-30 to 2016-04-27 ===
WRMSSE: 0.8470794036319829 | BIAS: 0.2203071817231114
quantile models training.

 === Window5: 2016-04-27 to 2016-05-22 ===
WRMSSE: 0.8472060763238827 | BIAS: 0.18195444597359772
quantile models training.


* After retraining every 4 weeks, we can see that the WRMSSE is stable across 4 windows, which signals that previously the inflation in WRMSSE
  was due to recursive error accumulating over the period, not model drift, not underlying pattern change.

* So the current model is good as when horizon is maximum 4 weeks. 

In [103]:
# NOW WITH 2 MONTHS 
results_28_new = stress_testing_expt(trained_models=models,train_df=final_train_data,test_df=test_data_price,
                                     features=unique_features,static_features=static_cols,window_length=28,
                                     retrain=False,cat_categories=cat_categories)


 === Window1: 2016-01-06 to 2016-02-03 ===
WRMSSE: 0.8443739790676126 | BIAS: 0.16565732956930068


In [110]:
results_no_retrain = [
    {"window": "W1 (Jan-Feb)",'date_start':'2016-01-06','date_end':'2016-02-03', "wrmsse": 0.8444, "bias": 0.1657},
    {"window": "W2 (Feb-Mar)",'date_start':'2016-02-03','date_end':'2016-03-02', "wrmsse": 0.9973, "bias": 0.3317},
    {"window": "W3 (Mar-Mar)", 'date_start':'2016-03-02','date_end':'2016-03-30',"wrmsse": 1.1099, "bias": 0.4156},
    {"window": "W4 (Mar-Apr)", 'date_start':'2016-03-30','date_end':'2016-04-27',"wrmsse": 1.2028, "bias": 0.4591},
    {"window": "W5 (Apr-May)", 'date_start':'2016-04-27','date_end':'2016-05-22',"wrmsse": 1.2721, "bias": 0.4827},
]

results_retrain = [
    {"window": "W1 (Jan-Feb)",'date_start':'2016-01-06','date_end':'2016-02-03', "wrmsse": 0.8444, "bias": 0.1657},
    {"window": "W2 (Feb-Mar)",'date_start':'2016-02-03','date_end':'2016-03-02', "wrmsse": 0.8652, "bias": 0.2297},
    {"window": "W3 (Mar-Mar)",'date_start':'2016-03-02','date_end':'2016-03-30', "wrmsse": 0.8419, "bias": 0.2268},
    {"window": "W4 (Mar-Apr)",'date_start':'2016-03-30','date_end':'2016-04-27', "wrmsse": 0.8471, "bias": 0.2203},
    {"window": "W5 (Apr-May)", 'date_start':'2016-04-27','date_end':'2016-05-22', "wrmsse": 0.8472, "bias": 0.1820},
]

df_no_retrain = pd.DataFrame(results_no_retrain)
df_retrain = pd.DataFrame(results_retrain)

# -------------------------------------------------------------------------
# 2. Performance Comparison Table
# -------------------------------------------------------------------------
comparison_df = pd.DataFrame(
    {
        "Window": df_retrain["window"],
        "WRMSSE (Static Model)": df_no_retrain["wrmsse"],
        "WRMSSE (Monthly Retrain)": df_retrain["wrmsse"],
        "Bias (Static Model)": df_no_retrain["bias"],
        "Bias (Monthly Retrain)": df_retrain["bias"],
    }
)

comparison_df

,Window,WRMSSE (Static Model),WRMSSE (Monthly Retrain),Bias (Static Model),Bias (Monthly Retrain)
0,W1 (Jan-Feb),0.8444,0.8444,0.1657,0.1657
1,W2 (Feb-Mar),0.9973,0.8652,0.3317,0.2297
2,W3 (Mar-Mar),1.1099,0.8419,0.4156,0.2268
3,W4 (Mar-Apr),1.2028,0.8471,0.4591,0.2203
4,W5 (Apr-May),1.2721,0.8472,0.4827,0.1820


In [113]:
# save the data
stress_expt_file = BASE_DIR/'results/stress_expt_results.csv'
comparison_df.to_csv(stress_expt_file)


### Empirical Coverage Check (Uncertainty Calibration)
In uncertainty/quantile forecasting, if your model outputs $q_{10}$ (10th percentile) and $q_{90}$ (90th percentile), the theoretical prediction interval spans $90\% - 10\% = 80\%$.To verify if your model is well-calibrated, at least 80% of actual sales values should fall inside the range $[q_{10}, q_{90}]$.

In [114]:
preds = results_28_new[2]['window_1']
preds

,item_id,date,sales_pred,q10,q90
0,FOODS_1_001,2016-01-06,0.749205,0.000000,2.352450
1,FOODS_1_002,2016-01-06,0.462250,0.000000,1.760795
2,FOODS_1_003,2016-01-06,0.716447,0.000000,2.272641
3,FOODS_1_004,2016-01-06,5.672297,1.064335,12.411240
4,FOODS_1_005,2016-01-06,2.026184,0.000000,4.128553
...,...,...,...,...,...
88377,HOUSEHOLD_2_512,2016-02-03,0.523636,0.000000,1.917292
88378,HOUSEHOLD_2_513,2016-02-03,0.451215,0.000000,1.359097
88379,HOUSEHOLD_2_514,2016-02-03,0.376451,0.000000,1.210480
88380,HOUSEHOLD_2_515,2016-02-03,0.201681,0.000000,1.062024


In [122]:
cutoff_date = test_data_price['date'].min()+pd.Timedelta(days=28)

test_df = test_data_price[test_data_price['date']<=cutoff_date]


test_df.reset_index()

,index,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,...,day_of_week,day_of_month,week_of_year,delta_price_weekn-1,historical_mean,historical_std,price_ratio_mean,is_discounted,dept_mean_price,delta_price_rltv_dept
0,731,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1804,0,2016-01-06,...,2,6,1,0.0,2.240234,0.0,1.0,0.0,3.325363,0.673681
1,732,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1805,0,2016-01-07,...,3,7,1,0.0,2.240234,0.0,1.0,0.0,3.325363,0.673681
2,733,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1806,3,2016-01-08,...,4,8,1,0.0,2.240234,0.0,1.0,0.0,3.325363,0.673681
3,734,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1807,0,2016-01-09,...,5,9,1,0.0,2.240234,0.0,1.0,0.0,3.329260,0.672893
4,735,FOODS_1_001_CA_1_evaluation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,d_1808,0,2016-01-10,...,6,10,1,0.0,2.240234,0.0,1.0,0.0,3.329260,0.672893
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88377,2575272,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1828,1,2016-01-30,...,5,30,4,0.0,5.941406,0.0,1.0,0.0,5.783844,1.027242
88378,2575273,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1829,1,2016-01-31,...,6,31,4,0.0,5.941406,0.0,1.0,0.0,5.783844,1.027242
88379,2575274,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1830,0,2016-02-01,...,0,1,5,0.0,5.941406,0.0,1.0,0.0,5.783844,1.027242
88380,2575275,HOUSEHOLD_2_516_CA_1_evaluation,HOUSEHOLD_2_516,HOUSEHOLD_2,HOUSEHOLD,CA_1,CA,d_1831,0,2016-02-02,...,1,2,5,0.0,5.941406,0.0,1.0,0.0,5.783844,1.027242


In [115]:


def calculate_prediction_coverage(
    df_test: pd.DataFrame, preds: pd.DataFrame
) -> float:
    """Calculates the empirical coverage percentage for 80% prediction intervals (q10 to q90).

    Parameters
    ----------
    df_test : pd.DataFrame
        DataFrame with ground truth sales. Must contain ['item_id', 'date', 'sales'].
    preds : pd.DataFrame
        DataFrame with model quantile predictions. Must contain ['item_id', 'date', 'q10', 'q90'].

    Returns
    -------
    float
        The proportion of actual sales that fall within [q10, q90] (0.0 to 1.0).
    """
    # 1. Merge actual sales with quantile predictions
    data = df_test[['item_id', 'date', 'sales']].merge(
        preds[['item_id', 'date', 'q10', 'q90']],
        on=['item_id', 'date'],
        how='left',
    )

    # 2. Check if actual sales fall within [q10, q90]
    data['is_covered'] = (data['sales'] >= data['q10']) & (
        data['sales'] <= data['q90']
    )

    # 3. Calculate overall coverage rate
    coverage_rate = data['is_covered'].mean()

    print(f"Empirical Coverage Rate: {coverage_rate * 100:.2f}%")
    if coverage_rate >= 0.80:
        print(" SUCCESS: Model meets or exceeds the 80% coverage target!")
    else:
        print(
            " WARNING: Model intervals are under-confident (narrower than expected)."
        )

    return coverage_rate

In [123]:
calculate_prediction_coverage(test_df,preds)

Empirical Coverage Rate: 89.42%
 SUCCESS: Model meets or exceeds the 80% coverage target!


np.float64(0.8941979136023173)

###  Holdout and Stockout cost on first 28 days of future data 


In [ ]:
from src.pipeline import cost_per_item 


costs = cost_per_item(test_df,preds)
costs

,item_id,total_cost
0,FOODS_1_001,0.414627
1,FOODS_1_002,0.325419
2,FOODS_1_003,4.759070
3,FOODS_1_004,21.237787
4,FOODS_1_005,1.725202
...,...,...
3044,HOUSEHOLD_2_512,6.282257
3045,HOUSEHOLD_2_513,0.031910
3046,HOUSEHOLD_2_514,4.925133
3047,HOUSEHOLD_2_515,0.018641


In [130]:

# save the result for the first 28 days

file =  BASE_DIR/"results/cost_per_items_2016_Jan_Feb.csv"

costs.to_csv(file)